# Pré-processamento dos dados de câncer de mama

In [1]:
# Pacotes utilizados
library(pacman)
p_load(haven, tidyverse, janitor, caret, recipes)

In [2]:
# Leitura dos dados
rm(list=ls(all=TRUE))
dados_brutos <- read_sav("../dados/dados_brutos.sav")

In [3]:
###  Tratamento de covariáveis 
dados_processados <- dados_brutos %>% select(c(RhcRacaCor, Instrução, EstCong, RhcHistoricoFamiliarCancer,
                             RhcHistoricoAlcool, RhcHistoricoTabaco, RhcOrigemEncamiamento,
                             RhcEstadiamentoClinico,RhcPrimeiroTratamentoRecebidoNoHospitalNenhum,
                             RhcPrimeiroTratamentoRecebidoNoHospitalCirurgia,
                             RhcPrimeiroTratamentoRecebidoNoHospitalHormonioterapia,
                             RhcPrimeiroTratamentoRecebidoNoHospitalOutras,
                             RhcPrimeiroTratamentoRecebidoNoHospitalRadioterapia,
                             RhcPrimeiroTratamentoRecebidoNoHospitalQuimioterapia,
                             # RhcPrimeiroTratamentoRecebidoNoHospitalTransplanteDeMedulaOssea, # 0 observações
                             # RhcPrimeiroTratamentoRecebidoNoHospitalImunoterapia, # apenas 4 
                             RhcDt1Consulta, RhcDtDiagnosticoTumor,RhcDt1TratamentoTumor,
                             RhcDiagnosticoETratamentosAnteriores,
                             RhcIdadeNoDiagnosticoTumor, 
                             Status,Tempo,
                             # CosemsCodIbge, # tem informação semelhante a macroregiao
                             Macroregião,
                             PndrRenda,
                             FaixaETCAT, Ano, 
                             TipoHCIDO )) %>%
  clean_names() %>%
  filter(tempo > 0) %>%
  mutate(
    id_paciente = row_number(),
    delta_t = as.numeric(case_when(
      status == "Morte por cancer de Mama" ~ 1,
      status == "Morte por outras causas" | status == "Viva"~ 0)),
    
    rhc_diagnostico_e_tratamentos_anteriores = factor(rhc_diagnostico_e_tratamentos_anteriores, 
                                                      levels = c("Com diag./Com trat.", "Com diag./Sem trat.", 
                                                                 "Sem diag./Sem trat."), exclude = NULL),
    
    tipo_hcido = factor(tipo_hcido) %>% 
      fct_lump_min(100, other_level = "min100"),
    
    
    diff_data_1consulta = as.numeric(difftime(rhc_dt1consulta, rhc_dt_diagnostico_tumor, units = "days")),
    
    diff_data_tratamento = as.numeric(difftime(rhc_dt1tratamento_tumor, rhc_dt_diagnostico_tumor, units = "days")),
    
    rhc_primeiro_tratamento_recebido_no_hospital_radioterapia = factor(case_when(
      rhc_primeiro_tratamento_recebido_no_hospital_radioterapia == "1" ~ "Sim",
      rhc_primeiro_tratamento_recebido_no_hospital_radioterapia == "0" ~ "Não"
    ), levels = c("Sim", "Não")),
    
    rhc_primeiro_tratamento_recebido_no_hospital_nenhum = factor(case_when(
      rhc_primeiro_tratamento_recebido_no_hospital_nenhum == "1" ~ "Sim",
      rhc_primeiro_tratamento_recebido_no_hospital_nenhum == "0" ~ "Não"
    ), levels = c("Sim", "Não")),
    
    rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia = factor(case_when(
      rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia == "1" ~ "Sim",
      rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia == "0" ~ "Não"
    ), levels = c("Sim", "Não")),
    
    rhc_primeiro_tratamento_recebido_no_hospital_cirurgia = factor(case_when(
      rhc_primeiro_tratamento_recebido_no_hospital_cirurgia == "1" ~ "Sim",
      rhc_primeiro_tratamento_recebido_no_hospital_cirurgia == "0" ~ "Não"
    ), levels = c("Sim", "Não")),
    
    rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia = factor(case_when(
      rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia == "1" ~ "Sim",
      rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia == "0" ~ "Não"
    ), levels = c("Sim", "Não")),
    
    
    
    rhc_primeiro_tratamento_recebido_no_hospital_outras = factor(case_when(
      rhc_primeiro_tratamento_recebido_no_hospital_outras == "1" ~ "Sim",
      rhc_primeiro_tratamento_recebido_no_hospital_outras == "0" ~ "Não"
    ), levels = c("Sim", "Não")),
    
    
    rhc_raca_cor = factor(rhc_raca_cor, 
                          levels = c("Branca", "Não Branca"), exclude = NULL),
    
    rhc_estadiamento_clinico = factor(rhc_estadiamento_clinico, 
                                      levels = c("I e II", "III e IV"), exclude = NULL),
    
    
    rhc_origem_encamiamento = factor(rhc_origem_encamiamento, 
                                     levels = c("SUS", "Não SUS"), exclude = NULL),
    
    
    rhc_historico_familiar_cancer = factor(rhc_historico_familiar_cancer,
                                           levels = c("Sim", "Não"), exclude = NULL),
    
    
    rhc_historico_tabaco = factor(rhc_historico_tabaco, 
                                  levels = c("Ex-consumidor", "Nunca", "Sim"), exclude = NULL),
    
    rhc_historico_alcool = factor(rhc_historico_alcool, 
                                  levels = c("Ex-consumidor", "Nunca", "Sim"), exclude = NULL),
    
    est_cong = factor(est_cong, levels = c(1,2), exclude = NULL),
    
    instrucao = factor(instrucao, levels = c(0,1,2), exclude = NULL),
    
    macroregiao = factor(macroregiao, levels = c(1,2,3), exclude = NULL),
    
    pndr_renda = factor(pndr_renda, levels = 
                          c("Alta Renda", "Baixa Renda", "Média Renda"), exclude = NULL),
    
    faixa_etcat = factor(faixa_etcat, exclude = NULL) 
  ) %>%
  select( 
    -c(status, rhc_dt_diagnostico_tumor, rhc_dt1consulta, rhc_dt1tratamento_tumor)
  )

In [4]:
dim(dados_processados)

[1] 6076   26

# Split dos dados em treino e teste e geração dos arquivos pré-processados.

In [5]:
### Criando amostras de treino e teste #######
set.seed(123)
linhas_treino <- createDataPartition(
  dados_processados$delta_t,
  p = 0.8,
  list = FALSE,
  times = 1
)

In [6]:
dados_processados_treino <- as.data.frame(dados_processados[linhas_treino, ])
dados_processados_teste <- as.data.frame(dados_processados[-linhas_treino, ])

In [7]:
### Imputando dados NAs para treino/teste/base completa
set.seed(123)
rec_imputacao <- recipe(tempo + delta_t ~ ., data = dados_processados_treino) %>%
  step_impute_knn(all_predictors()) %>%
  prep(training = dados_processados_treino)

dados_processados_treino_imputado <- bake(rec_imputacao, new_data = dados_processados_treino)
dados_processados_teste_imputado <- bake(rec_imputacao, new_data = dados_processados_teste)
dados_processados_imputado <- bake(rec_imputacao, new_data = dados_processados)

In [8]:
# Gerar os arquivos .csv

write.csv(dados_processados_treino_imputado, "../dados/dados-processados/dados_treino.csv", row.names = FALSE)
write.csv(dados_processados_teste_imputado, "../dados/dados-processados/dados_teste.csv", row.names = FALSE)
write.csv(dados_processados_imputado, "../dados/dados-processados/dados_processados.csv", row.names = FALSE)